In [1]:
# Setup: DuckDB — zero config, runs in-process, full SQL
# pip install duckdb
import duckdb

# Create an in-memory connection
con = duckdb.connect()

# -------------------------------------------------------
# Seed data: Citi-style server telemetry (employees for ranking demos)
# -------------------------------------------------------

def sql_executer(str):
    con.execute(str)

def sql_executer_printer(str):
    print (con.execute(str).df().to_string(index=False))   

In [2]:
sql = """
DROP TABLE IF EXISTS employees;
CREATE TABLE  employees (
    employee_id INTEGER PRIMARY KEY, name TEXT, department_id INTEGER, salary REAL
);
INSERT INTO employees VALUES
    (1,'Alice',1,95000),(2,'Bob',1,72000),(3,'Carol',1,88000),
    (4,'Dave',2,105000),(5,'Eve',2,98000),(6,'Frank',2,67000),
    (7,'Grace',3,82000),(8,'Hank',3,79000),(9,'Iris',3,91000),
    (10,'Jack',3,55000);
DROP TABLE IF EXISTS telemetry_raw;
CREATE TABLE telemetry_raw (
    server_id TEXT, collection_date TEXT, cpu_utilization REAL
);
INSERT INTO telemetry_raw VALUES
    ('srv-01','2026-02-01',72.5),('srv-01','2026-02-01',73.1),
    ('srv-01','2026-02-02',85.2),('srv-01','2026-02-03',91.0),
    ('srv-01','2026-02-04',88.7),('srv-01','2026-02-05',95.3),
    ('srv-02','2026-02-01',45.0),('srv-02','2026-02-02',47.2),
    ('srv-02','2026-02-03',50.1),('srv-02','2026-02-04',48.8),
    ('srv-02','2026-02-05',46.5),('srv-03','2026-02-01',78.0),
    ('srv-03','2026-02-02',82.3),('srv-03','2026-02-03',85.1),
    ('srv-03','2026-02-04',88.9),('srv-03','2026-02-05',92.4);
DROP TABLE IF EXISTS employees_org;
CREATE TABLE employees_org (
    employee_id INTEGER, name TEXT, manager_id INTEGER
);
INSERT INTO employees_org VALUES
    (1,'CEO',NULL),(2,'CTO',1),(3,'CFO',1),
    (4,'VP Engineering',2),(5,'VP Data',2),
    (6,'Senior DE',4),(7,'Data Engineer',5),(8,'Analyst',5);
"""
sql_executer(sql)

In [5]:
sql = """
WITH dept_averages AS (
    SELECT
        department_id,
        AVG(salary) AS avg_salary
    FROM employees
    GROUP BY department_id
)
SELECT
    e.name,
    e.salary,
    ROUND(d.avg_salary, 2) AS dept_avg,
    ROUND(e.salary - d.avg_salary, 2) AS above_avg_by
FROM employees e
JOIN dept_averages d ON e.department_id = d.department_id
WHERE e.salary > d.avg_salary
ORDER BY above_avg_by DESC
"""
print("Employees earning above their department average (CTE version):")
print("\nSame result, subquery version (harder to read and reuse):")
sql_executer_printer(sql)

Employees earning above their department average (CTE version):

Same result, subquery version (harder to read and reuse):
 name   salary  dept_avg  above_avg_by
 Dave 105000.0   90000.0       15000.0
 Iris  91000.0   76750.0       14250.0
Alice  95000.0   85000.0       10000.0
  Eve  98000.0   90000.0        8000.0
Grace  82000.0   76750.0        5250.0
Carol  88000.0   85000.0        3000.0
 Hank  79000.0   76750.0        2250.0


In [7]:
# Capacity planning pipeline — chained CTEs (conn from c00-setup)
# DuckDB-compatible: uses CURRENT_DATE - INTERVAL instead of SQLite's date()

sql_pipeline = """
WITH deduped AS (
    -- Remove any duplicate records from the same server/day
    SELECT DISTINCT
        server_id,
        collection_date,
        cpu_utilization
    FROM telemetry_raw
    WHERE CAST(collection_date AS DATE) >= CURRENT_DATE - INTERVAL '365' DAY
),
daily_avg AS (
    -- One row per server per day
    SELECT
        server_id,
        collection_date,
        AVG(cpu_utilization) AS avg_cpu,
        MAX(cpu_utilization) AS peak_cpu
    FROM deduped
    GROUP BY server_id, collection_date
),
monthly_stats AS (
    -- Aggregate to summary per server
    SELECT
        server_id,
        AVG(avg_cpu)  AS avg_30d,
        MAX(peak_cpu) AS peak_30d,
        COUNT(*)      AS days_sampled
    FROM daily_avg
    GROUP BY server_id
)
SELECT
    server_id,
    ROUND(avg_30d, 2)  AS avg_cpu_30d,
    ROUND(peak_30d, 2) AS peak_cpu_30d,
    days_sampled,
    CASE WHEN peak_30d > 80 THEN 'AT RISK'
         WHEN avg_30d  > 65 THEN 'MONITOR'
         ELSE 'HEALTHY' END AS status
FROM monthly_stats
ORDER BY peak_30d DESC
"""
print("Capacity planning pipeline result:")
sql_executer_printer(sql_pipeline)
print("\nTip: Test any step in isolation — replace final SELECT with: SELECT * FROM daily_avg")


Capacity planning pipeline result:
server_id  avg_cpu_30d  peak_cpu_30d  days_sampled  status
   srv-01        86.60     95.300003             5 AT RISK
   srv-03        85.34     92.400002             5 AT RISK
   srv-02        47.52     50.099998             5 HEALTHY

Tip: Test any step in isolation — replace final SELECT with: SELECT * FROM daily_avg


In [11]:
# Recursive CTE: Walk org chart from CEO down (conn from c00-setup)
# SQLite supports RECURSIVE CTEs. Uses substr() for indent (no LPAD in SQLite).

sql_recursive = """
WITH RECURSIVE org_chart AS (
    -- BASE CASE: The root node (CEO has no manager)
    SELECT
        employee_id,
        name,
        manager_id,
        1 AS org_level,
        name AS path
    FROM employees_org
    WHERE manager_id IS NULL

    UNION ALL

    -- RECURSIVE CASE: Join each employee to their manager row
    SELECT
        e.employee_id,
        e.name,
        e.manager_id,
        oc.org_level + 1,
        oc.path || ' > ' || e.name
    FROM employees_org e
    JOIN org_chart oc ON e.manager_id = oc.employee_id
    WHERE oc.org_level < 10
)
SELECT
    name AS org_tree,
    org_level,
    path
FROM org_chart
ORDER BY path
"""
sql_executer_printer(sql_recursive)

      org_tree  org_level                                   path
           CEO          1                                    CEO
           CFO          2                              CEO > CFO
           CTO          2                              CEO > CTO
       VP Data          3                    CEO > CTO > VP Data
       Analyst          4          CEO > CTO > VP Data > Analyst
 Data Engineer          4    CEO > CTO > VP Data > Data Engineer
VP Engineering          3             CEO > CTO > VP Engineering
     Senior DE          4 CEO > CTO > VP Engineering > Senior DE
